## Lab 3 (Weeks 6–8): Support Case Analytics, Organizational Hierarchies, and Advanced Dimensional Modeling

> Before starting, review these resources:
> - `lab3_intro.md` — concepts covered in lecture
> - `hierarchy_intro.md` — how ragged hierarchies work and how to populate a bridge table (required for Task 5)

### What you are building
A brand-new business process — **customer support** — built on top of the dimensional infrastructure from Labs 1 and 2.

You will:
- Reuse `dim_account` and `dim_date` as conformed dimensions (no changes needed)
- Create `dim_support_case` (Type 1), `dim_user` (Type 2), `dim_contact` (Type 2)
- Build a bridge table for a ragged contact/manager organizational hierarchy
- Design and load `fact_support_case`

### Naming convention
All table names must end with your `_UNDERWOOD_J` suffix.
Save this notebook as `UNDERWOOD_J_LAB3` in the `MODELED` schema (e.g., `CLARK_C_LAB3`).

---
## Setup

In [1]:
USE DATABASE SNOWBEARAIR_DB;
USE SCHEMA MODELED;
USE WAREHOUSE STUDENT_WH;

status
Statement executed successfully.


status
Statement executed successfully.


status
Statement executed successfully.


---
## Explore the Source Tables

Before designing anything, run these to understand the data. Pay attention to:
- What fields does `support_case` have? What are its date fields?
- How does `contact` relate to `salesforce_user`?
- What is `manager_id` in `contact` — does it point back to `contact`?

In [2]:
SELECT * FROM SNOWBEARAIR_DB.PUBLIC.support_case     LIMIT 10;
-- SELECT * FROM SNOWBEARAIR_DB.PUBLIC.contact         LIMIT 10;
-- SELECT * FROM SNOWBEARAIR_DB.PUBLIC.salesforce_user LIMIT 10;

ID,TYPE,STATUS,PRIORITY,ACCOUNT_ID,CONTACT_ID,OWNER_ID,OPENED_DATE,CLOSED_DATE
SC001,Problem,Closed,High,A001,C001,U003,2022-04-01,2022-04-03
SC002,Question,In Progress,Medium,A002,C007,U003,2022-04-10,NULL


---
## Task 1: Declare the Business Process and Grain

Before writing any SQL, answer these questions:
1. What business process are we measuring?
2. What does one row in `fact_support_case` represent?

**Business Process:**
Customer support: tracking support cases and how long they take to resolve.

**Grain Statement:**
One row per support case, linked to its account, support agent, contact, and opening and closing dates.

---
# Part 1: Design — Create Your Tables

**Build order (dependencies matter):**
1. `dim_support_case` (Type 1)
2. `dim_user` (Type 2)
3. `dim_contact` (Type 2)
4. `dim_management_hierarchy_bridge`
5. `fact_support_case`

**Conformed dimensions from prior labs** — use these directly, no changes needed:
- `dim_account_underwood_j` (from Lab 2, Type 2)
- `dim_date_underwood_j` (from Lab 2, Type 0)

---
### Task 2: Create dim_support_case (Type 1)

Create `dim_support_case_underwood_j`

**Requirements:**
- `support_case_key` (AUTOINCREMENT)
- `case_id` natural key
- Descriptive attributes: case type, case status, case priority
- **Type 1 SCD** — no effective dates or current flag needed

**Why Type 1?** Status and priority reflect the current operational state. Analysts want to see whether a case is currently Open or Closed — not a history of every status transition.

In [3]:
-- Task 2: Create dim_support_case_underwood_j (Type 1)

CREATE OR REPLACE TABLE dim_support_case_underwood_j (
    support_case_key INTEGER AUTOINCREMENT PRIMARY KEY,
    case_id          VARCHAR(255) NOT NULL,
    case_type        VARCHAR(100),
    case_status      VARCHAR(100),
    case_priority    VARCHAR(100)
);

status
Table DIM_SUPPORT_CASE_UNDERWOOD_J successfully created.


---
### Task 3: Create dim_user (Type 2)

Create `dim_user_underwood_j`

**Requirements:**
- `user_key` (AUTOINCREMENT)
- `user_id` natural key
- Descriptive attributes: full name, title, department, role, active indicator
- **Type 2 SCD**: `effective_start_date`, `effective_end_date` (default `'9999-12-31'`), `current_flag`

**Why Type 2?** When a support agent changes departments or roles, historical cases must still show the correct organizational context at the time they were worked.

In [4]:
-- Task 3: Create dim_user_underwood_j (Type 2)

CREATE OR REPLACE TABLE dim_user_underwood_j (
    user_key             INTEGER AUTOINCREMENT PRIMARY KEY,
    user_id              VARCHAR(255) NOT NULL,
    full_name            VARCHAR(255),
    title                VARCHAR(255),
    department           VARCHAR(255),
    role                 VARCHAR(255),
    is_active            BOOLEAN,
    effective_start_date DATE NOT NULL,
    effective_end_date   DATE NOT NULL DEFAULT '9999-12-31'::DATE,
    current_flag         BOOLEAN NOT NULL DEFAULT TRUE
);

status
Table DIM_USER_UNDERWOOD_J successfully created.


---
### Task 4: Create dim_contact (Type 2)

Create `dim_contact_underwood_j`

**Requirements:**
- `contact_key` (AUTOINCREMENT)
- `contact_id` natural key
- Descriptive attributes: first name, last name, email, title, department
- `manager_contact_id` — the **natural key** of this contact's manager (used for the hierarchy in Task 5)
- **Type 2 SCD**: `effective_start_date`, `effective_end_date`, `current_flag`

In [5]:
-- Task 4: Create dim_contact_underwood_j (Type 2)

CREATE OR REPLACE TABLE dim_contact_underwood_j (
    contact_key          INTEGER AUTOINCREMENT PRIMARY KEY,
    contact_id           VARCHAR(255) NOT NULL,
    first_name           VARCHAR(255),
    last_name            VARCHAR(255),
    email                VARCHAR(255),
    title                VARCHAR(255),
    department           VARCHAR(255),
    manager_contact_id   VARCHAR(255),
    account_id           VARCHAR(255),
    effective_start_date DATE NOT NULL,
    effective_end_date   DATE NOT NULL DEFAULT '9999-12-31'::DATE,
    current_flag         BOOLEAN NOT NULL DEFAULT TRUE
);

status
Table DIM_CONTACT_UNDERWOOD_J successfully created.


---
### Task 5: Create dim_management_hierarchy_bridge

Create `dim_management_hierarchy_bridge_underwood_j`

This bridge table stores **every subordinate-ancestor pair** in the org chart, regardless of depth. That's what makes it handle a ragged (variable-depth) hierarchy.

**Requirements:**
- `contact_key` — the subordinate (FK to `dim_contact_underwood_j`)
- `manager_contact_key` — any ancestor at any level above (FK to `dim_contact_underwood_j`)
- `levels_between` — 0 = self, 1 = direct manager, 2 = skip-level, etc.
- `top_flag` — TRUE if the manager_contact_key row is the top of the hierarchy
- `effective_start_date`, `effective_end_date`, `current_flag`

**Why a bridge table?** Not all contacts have the same number of reporting levels. A bridge table stores all relationships explicitly — no fixed-depth columns needed.

In [6]:
-- Task 5: Create dim_management_hierarchy_bridge_underwood_j

CREATE OR REPLACE TABLE dim_management_hierarchy_bridge_underwood_j (
    hierarchy_bridge_key INTEGER AUTOINCREMENT PRIMARY KEY,
    contact_key          INTEGER NOT NULL,
    manager_contact_key  INTEGER NOT NULL,
    levels_between       INTEGER NOT NULL,
    top_flag             BOOLEAN NOT NULL,
    effective_start_date DATE NOT NULL,
    effective_end_date   DATE NOT NULL DEFAULT '9999-12-31'::DATE,
    current_flag         BOOLEAN NOT NULL DEFAULT TRUE,
    UNIQUE (contact_key, manager_contact_key, effective_start_date),
    FOREIGN KEY (contact_key) REFERENCES dim_contact_underwood_j(contact_key),
    FOREIGN KEY (manager_contact_key) REFERENCES dim_contact_underwood_j(contact_key)
);

status
Table DIM_MANAGEMENT_HIERARCHY_BRIDGE_UNDERWOOD_J successfully created.


---
### Task 6: Create the Fact Table

Create `fact_support_case_underwood_j`

**Requirements:**
- `case_fact_key` (AUTOINCREMENT PK)
- Foreign keys: `account_key`, `support_case_key`, `user_key`, `contact_key`
- Role-playing date keys: `opened_date_key` and `closed_date_key` — both reference `dim_date_underwood_j`
- Measure: `resolution_days` (INTEGER, NULL if case still open)

> The fact table must contain **only surrogate keys and measures** — no descriptive text, no natural identifiers.

In [7]:
-- Task 6: Create fact_support_case_underwood_j

CREATE OR REPLACE TABLE fact_support_case_underwood_j (
    case_fact_key     INTEGER AUTOINCREMENT PRIMARY KEY,
    account_key       INTEGER NOT NULL,
    support_case_key  INTEGER NOT NULL,
    user_key          INTEGER NOT NULL,
    contact_key       INTEGER NOT NULL,
    opened_date_key   INTEGER NOT NULL,
    closed_date_key   INTEGER,
    case_count        INTEGER NOT NULL DEFAULT 1,
    resolution_days   INTEGER,
    UNIQUE (support_case_key),
    FOREIGN KEY (account_key) REFERENCES dim_account_underwood_j(account_key),
    FOREIGN KEY (support_case_key) REFERENCES dim_support_case_underwood_j(support_case_key),
    FOREIGN KEY (user_key) REFERENCES dim_user_underwood_j(user_key),
    FOREIGN KEY (contact_key) REFERENCES dim_contact_underwood_j(contact_key),
    FOREIGN KEY (opened_date_key) REFERENCES dim_date_underwood_j(date_key),
    FOREIGN KEY (closed_date_key) REFERENCES dim_date_underwood_j(date_key)
);

status
Table FACT_SUPPORT_CASE_UNDERWOOD_J successfully created.


---
# Part 2: Load Data

**Load order:** dimensions → bridge → fact

---
### Load dim_support_case (Type 1)

In [8]:
-- Load dim_support_case_underwood_j

INSERT INTO dim_support_case_underwood_j (
    case_id, case_type, case_status, case_priority
)
SELECT
    id, type, status, priority
FROM SNOWBEARAIR_DB.PUBLIC.support_case;

number of rows inserted
2


---
### Load dim_user (Type 2 initial load)

Set `effective_start_date = CURRENT_DATE`, `effective_end_date = '9999-12-31'`, `current_flag = TRUE` for all rows on initial load.

In [9]:
-- Load dim_user_underwood_j (Type 2 initial load)

INSERT INTO dim_user_underwood_j (
    user_id, full_name, title, department, role, is_active,
    effective_start_date, effective_end_date, current_flag
)
SELECT
    id, name, title, department, user_role, is_active,
    CURRENT_DATE(), '9999-12-31'::DATE, TRUE
FROM SNOWBEARAIR_DB.PUBLIC.salesforce_user;

number of rows inserted
3


---
### Load dim_contact (Type 2 initial load)

In [10]:
-- Load dim_contact_underwood_j (Type 2 initial load)

INSERT INTO dim_contact_underwood_j (
    contact_id, first_name, last_name, email, title, department,
    manager_contact_id, account_id,
    effective_start_date, effective_end_date, current_flag
)
SELECT
    c.id, c.first_name, c.last_name, c.email, c.title, c.department,
    c.manager_id, a.account_id,
    CURRENT_DATE(), '9999-12-31'::DATE, TRUE
FROM SNOWBEARAIR_DB.PUBLIC.contact c
-- The source has no account_id; use a link only when cases identify one account.
LEFT JOIN (
    SELECT contact_id, MAX(account_id) AS account_id
    FROM SNOWBEARAIR_DB.PUBLIC.support_case
    WHERE account_id IS NOT NULL
    GROUP BY contact_id
    HAVING COUNT(DISTINCT account_id) = 1
) a ON a.contact_id = c.id;

number of rows inserted
12


---
### Load dim_management_hierarchy_bridge

The bridge needs one row per subordinate-ancestor pair:
- A row for each contact pointing to themselves (levels_between = 0)
- A row for each contact pointing to their direct manager (levels_between = 1)

**Join pattern:** self-join `dim_contact_underwood_j` on `manager_contact_id = contact_id`.

In [11]:
-- Load dim_management_hierarchy_bridge_underwood_j
-- Two-step approach: insert self rows first (levels_between = 0),
-- then insert direct manager rows (levels_between = 1) using a self-join on dim_contact.

INSERT INTO dim_management_hierarchy_bridge_underwood_j (
    contact_key, manager_contact_key, levels_between, top_flag,
    effective_start_date, effective_end_date, current_flag
)
SELECT
    contact_key, contact_key, 0, manager_contact_id IS NULL,
    CURRENT_DATE(), '9999-12-31'::DATE, TRUE
FROM dim_contact_underwood_j
WHERE current_flag = TRUE
;

-- Start with direct managers, then follow their managers to every higher level.
INSERT INTO dim_management_hierarchy_bridge_underwood_j (
    contact_key, manager_contact_key, levels_between, top_flag,
    effective_start_date, effective_end_date, current_flag
)
WITH RECURSIVE current_contacts AS (
    SELECT contact_key, contact_id, manager_contact_id
    FROM dim_contact_underwood_j
    WHERE current_flag = TRUE
), manager_paths (contact_key, manager_contact_key, levels_between,
                  next_manager_id, visited_contacts) AS (
    SELECT c.contact_key, m.contact_key, 1::INTEGER,
           m.manager_contact_id, ARRAY_CONSTRUCT(c.contact_id, m.contact_id)
    FROM current_contacts c
    JOIN current_contacts m ON m.contact_id = c.manager_contact_id
    WHERE c.contact_id != m.contact_id

    UNION ALL

    SELECT p.contact_key, m.contact_key, p.levels_between + 1,
           m.manager_contact_id,
           ARRAY_APPEND(p.visited_contacts, TO_VARIANT(m.contact_id))
    FROM manager_paths p
    JOIN current_contacts m ON m.contact_id = p.next_manager_id
    WHERE NOT ARRAY_CONTAINS(TO_VARIANT(m.contact_id), p.visited_contacts)
)
SELECT contact_key, manager_contact_key, levels_between,
       next_manager_id IS NULL,
       CURRENT_DATE(), '9999-12-31'::DATE, TRUE
FROM manager_paths;


number of rows inserted
12


number of rows inserted
18


---
### Load fact_support_case

Join through dimension tables to resolve surrogate keys.

- Use the date key resolution pattern from Lab 2 for `opened_date_key` and `closed_date_key`
- Calculate `resolution_days` from the open and close date columns (NULL if the case is still open)
- When joining Type 2 dimensions, filter to the current row

In [12]:
-- Load fact_support_case_underwood_j
-- Join through dimension tables to resolve surrogate keys.
-- Use the date key resolution and role-playing join pattern from Lab 2.

INSERT INTO fact_support_case_underwood_j (
    account_key, support_case_key, user_key, contact_key,
    opened_date_key, closed_date_key, case_count, resolution_days
)
SELECT
    a.account_key, sc.support_case_key, u.user_key, c.contact_key,
    opened.date_key, closed.date_key, 1,
    DATEDIFF(day, src.opened_date, src.closed_date)
FROM SNOWBEARAIR_DB.PUBLIC.support_case src
JOIN dim_account_underwood_j a
    ON a.account_id = src.account_id AND a.current_flag = TRUE
JOIN dim_support_case_underwood_j sc ON sc.case_id = src.id
JOIN dim_user_underwood_j u
    ON u.user_id = src.owner_id AND u.current_flag = TRUE
JOIN dim_contact_underwood_j c
    ON c.contact_id = src.contact_id AND c.current_flag = TRUE
JOIN dim_date_underwood_j opened
    ON opened.date_key = TO_NUMBER(TO_CHAR(src.opened_date, 'YYYYMMDD'))
LEFT JOIN dim_date_underwood_j closed
    ON closed.date_key = TO_NUMBER(TO_CHAR(src.closed_date, 'YYYYMMDD'))
;

number of rows inserted
2


---
## Verify Your Work

In [13]:
-- Row counts
SELECT 'dim_support_case'                   AS table_name, COUNT(*) AS row_count FROM dim_support_case_underwood_j
UNION ALL SELECT 'dim_user',                              COUNT(*)              FROM dim_user_underwood_j
UNION ALL SELECT 'dim_contact',                           COUNT(*)              FROM dim_contact_underwood_j
UNION ALL SELECT 'dim_management_hierarchy_bridge',       COUNT(*)              FROM dim_management_hierarchy_bridge_underwood_j
UNION ALL SELECT 'fact_support_case',                     COUNT(*)              FROM fact_support_case_underwood_j;

-- Cases by priority and status
SELECT sc.case_priority, sc.case_status, COUNT(*), AVG(f.resolution_days)
FROM fact_support_case_underwood_j f
JOIN dim_support_case_underwood_j sc ON sc.support_case_key = f.support_case_key
GROUP BY sc.case_priority, sc.case_status;

-- Roll up cases to any manager level using bridge table
SELECT mgr.first_name || ' ' || mgr.last_name AS manager, b.levels_between, COUNT(*) AS cases
FROM fact_support_case_underwood_j f
JOIN dim_contact_underwood_j c     ON c.contact_key = f.contact_key
JOIN dim_management_hierarchy_bridge_underwood_j b ON b.contact_key = c.contact_key AND b.current_flag = TRUE
JOIN dim_contact_underwood_j mgr   ON mgr.contact_key = b.manager_contact_key AND mgr.current_flag = TRUE
GROUP BY mgr.first_name, mgr.last_name, b.levels_between
ORDER BY manager, b.levels_between;

-- Expected counts and key checks for the supplied class dataset.
WITH checks AS (
    SELECT 1 AS check_order, 'Case descriptions' AS check_name,
           COUNT(*) AS actual, 2 AS expected FROM dim_support_case_underwood_j
    UNION ALL SELECT 2, 'Users', COUNT(*), 3 FROM dim_user_underwood_j
    UNION ALL SELECT 3, 'Contacts', COUNT(*), 12 FROM dim_contact_underwood_j
    UNION ALL SELECT 4, 'Hierarchy paths', COUNT(*), 30 FROM dim_management_hierarchy_bridge_underwood_j
    UNION ALL SELECT 5, 'Self paths', COUNT(*), 12 FROM dim_management_hierarchy_bridge_underwood_j WHERE levels_between = 0
    UNION ALL SELECT 6, 'Maximum hierarchy depth', MAX(levels_between), 3 FROM dim_management_hierarchy_bridge_underwood_j
    UNION ALL SELECT 7, 'Fact cases', COUNT(*), 2 FROM fact_support_case_underwood_j
    UNION ALL SELECT 8, 'Case count measure', SUM(case_count), 2 FROM fact_support_case_underwood_j
    UNION ALL SELECT 9, 'Closed cases', COUNT(closed_date_key), 1 FROM fact_support_case_underwood_j
    UNION ALL SELECT 10, 'Closed resolution days', SUM(resolution_days), 2 FROM fact_support_case_underwood_j
    UNION ALL SELECT 11, 'Duplicate case natural keys', COUNT(*), 0
        FROM (SELECT case_id FROM dim_support_case_underwood_j GROUP BY case_id HAVING COUNT(*) > 1)
    UNION ALL SELECT 12, 'Duplicate fact cases', COUNT(*), 0
        FROM (SELECT support_case_key FROM fact_support_case_underwood_j GROUP BY support_case_key HAVING COUNT(*) > 1)
    UNION ALL SELECT 13, 'Duplicate hierarchy pairs', COUNT(*), 0
        FROM (SELECT contact_key, manager_contact_key FROM dim_management_hierarchy_bridge_underwood_j
              GROUP BY contact_key, manager_contact_key HAVING COUNT(*) > 1)
    UNION ALL SELECT 14, 'Fact dimension orphans', COUNT(*), 0
        FROM fact_support_case_underwood_j f
        LEFT JOIN dim_account_underwood_j a ON a.account_key = f.account_key
        LEFT JOIN dim_support_case_underwood_j sc ON sc.support_case_key = f.support_case_key
        LEFT JOIN dim_user_underwood_j u ON u.user_key = f.user_key
        LEFT JOIN dim_contact_underwood_j c ON c.contact_key = f.contact_key
        LEFT JOIN dim_date_underwood_j opened ON opened.date_key = f.opened_date_key
        LEFT JOIN dim_date_underwood_j closed ON closed.date_key = f.closed_date_key
        WHERE a.account_key IS NULL OR sc.support_case_key IS NULL OR u.user_key IS NULL
           OR c.contact_key IS NULL OR opened.date_key IS NULL
           OR (f.closed_date_key IS NOT NULL AND closed.date_key IS NULL)
    UNION ALL SELECT 15, 'Bridge dimension orphans', COUNT(*), 0
        FROM dim_management_hierarchy_bridge_underwood_j b
        LEFT JOIN dim_contact_underwood_j c ON c.contact_key = b.contact_key
        LEFT JOIN dim_contact_underwood_j m ON m.contact_key = b.manager_contact_key
        WHERE c.contact_key IS NULL OR m.contact_key IS NULL
    UNION ALL SELECT 16, 'Users without one current profile', COUNT(*), 0
        FROM (SELECT user_id FROM dim_user_underwood_j GROUP BY user_id HAVING COUNT_IF(current_flag) != 1)
    UNION ALL SELECT 17, 'Contacts without one current profile', COUNT(*), 0
        FROM (SELECT contact_id FROM dim_contact_underwood_j GROUP BY contact_id HAVING COUNT_IF(current_flag) != 1)
    UNION ALL SELECT 18, 'Case measures or dates not matching source', COUNT(*), 0
        FROM fact_support_case_underwood_j f
        JOIN dim_support_case_underwood_j sc ON sc.support_case_key = f.support_case_key
        JOIN SNOWBEARAIR_DB.PUBLIC.support_case src ON src.id = sc.case_id
        JOIN dim_date_underwood_j opened ON opened.date_key = f.opened_date_key
        LEFT JOIN dim_date_underwood_j closed ON closed.date_key = f.closed_date_key
        WHERE opened.calendar_date IS DISTINCT FROM src.opened_date
           OR closed.calendar_date IS DISTINCT FROM src.closed_date
           OR f.resolution_days IS DISTINCT FROM DATEDIFF(day, src.opened_date, src.closed_date)
    UNION ALL SELECT 19, 'Unresolved manager IDs', COUNT(*), 0
        FROM dim_contact_underwood_j c
        LEFT JOIN dim_contact_underwood_j m ON m.contact_id = c.manager_contact_id AND m.current_flag = TRUE
        WHERE c.manager_contact_id IS NOT NULL AND m.contact_key IS NULL
    UNION ALL SELECT 20, 'Shared accounts', COUNT(*), 3 FROM dim_account_underwood_j
    UNION ALL SELECT 21, 'Shared dates', COUNT(*), 5844 FROM dim_date_underwood_j
)
SELECT check_name, actual, expected,
       IFF(actual = expected, 'PASS', 'FAIL') AS result
FROM checks
ORDER BY check_order;


TABLE_NAME,ROW_COUNT
dim_support_case,2
dim_user,3
dim_contact,12
dim_management_hierarchy_bridge,30
fact_support_case,2


CASE_PRIORITY,CASE_STATUS,COUNT(*),AVG(F.RESOLUTION_DAYS)
High,Closed,1,2.000000
Medium,In Progress,1,NULL


MANAGER,LEVELS_BETWEEN,CASES
Karen Martinez,1,1
Michael Johnson,0,1
Robert Taylor,0,1


CHECK_NAME,ACTUAL,EXPECTED,RESULT
Case descriptions,2,2,PASS
Users,3,3,PASS
Contacts,12,12,PASS
Hierarchy paths,30,30,PASS
Self paths,12,12,PASS
Maximum hierarchy depth,3,3,PASS
Fact cases,2,2,PASS
Case count measure,2,2,PASS
Closed cases,1,1,PASS
Closed resolution days,2,2,PASS


---
## Deliverables Checklist

- [ ] Notebook saved as `UNDERWOOD_J_LAB3` in the `MODELED` schema
- [x] All table names include your `_UNDERWOOD_J` suffix
- [x] Task 1 grain answer written in the markdown cell above
- [x] All 5 DDL cells run successfully (Tasks 2–6)
- [x] All load cells run without errors
- [x] Bridge table has at least 2× rows vs contact count (self + manager rows)
- [x] Row counts verified
- [x] Star schema diagram uploaded **or** full DDL written (either/or)
- [ ] Written explanation in Canvas covering:
  - Fact grain justification
  - SCD strategy rationale for each dimension
  - Ragged hierarchy design: why a bridge table and how you'd query it
  - At least one modeling tradeoff you considered